# Phase 0: pretrained-backbone deployment path check (NeMo -> ONNX)

Continues `docs/plans/audio_eval_notebook_refactor_plan.md`, "Pretrained Backbone / Transfer Learning Track".

Before spending any effort fine-tuning a pretrained speech-command model on our dataset, verify the part most likely to hide a nasty surprise: can a stock NeMo MatchboxNet checkpoint actually be loaded and exported to ONNX at all? This notebook does nothing with our own dataset -- it's purely a feasibility check on the pretrained checkpoint and the export path, run in Colab because NeMo has several Linux-only dependencies (`pynini`, `nemo_text_processing`, etc.) that make it painful to install natively on Windows.

Confirmed before writing this (via NVIDIA's own docs/NGC pages, not guessed):
- `commandrecognition_en_matchboxnet3x1x64_v2` / `commandrecognition_en_matchboxnet3x2x64_v2` are real, loadable via `EncDecClassificationModel.from_pretrained(...)`
- MatchboxNet 3x2x1-scale checkpoints run ~93K parameters, ~767KB compressed, 97.3% accuracy on Speech Commands v2 (35 classes) -- comfortably edge-appropriate, and still ~7x more capacity than our current 13.5K-param custom CNN
- NeMo models export to ONNX via the `Exportable` mixin's `.export()` method; ONNX -> TensorRT is NVIDIA's own documented Jetson deployment path

What this notebook actually verifies, that the docs alone don't prove:
1. The checkpoint downloads and loads without version/dependency drift breaking it
2. What the model actually expects as input (sample rate, class count/labels) -- needed to design the fine-tuning data pipeline correctly
3. `.export()` produces a working ONNX file, not just a documented method that exists
4. The exported ONNX graph actually runs in `onnxruntime` and produces sane output shapes

If any of this breaks, better to find out now than after a fine-tuning investment.

In [ ]:
# NeMo's ASR/classification collection pulls in a lot -- this can take a few minutes.
!pip install -q "nemo_toolkit[asr]"
!pip install -q onnx onnxruntime

In [ ]:
import nemo.collections.asr as nemo_asr
import torch

print("NeMo import OK")
print("CUDA available:", torch.cuda.is_available())

## Load the smallest pretrained checkpoint and inspect what it actually expects

Starting with `3x1x64` (the smaller of the two confirmed variants) since edge footprint is the whole point. If this loads cleanly, also try `3x2x64` for the accuracy/size comparison.

In [ ]:
model = nemo_asr.models.EncDecClassificationModel.from_pretrained(
    model_name="commandrecognition_en_matchboxnet3x1x64_v2"
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameter count: {n_params:,}")
print(f"Pretrained labels ({len(model.cfg.labels)}): {model.cfg.labels}")
print(f"Expected sample rate: {model.cfg.train_ds.get('sample_rate', model.cfg.get('sample_rate', 'unknown'))}")
print("Preprocessor cfg:", model.cfg.preprocessor)

## Sanity inference on a dummy waveform

Confirms the forward pass works end-to-end (preprocessor -> encoder -> decoder) before trusting anything downstream. 1 second of silence/noise at the model's expected sample rate -- output doesn't need to be meaningful, just needs to run and produce a (1, num_classes) logits tensor.

In [ ]:
sample_rate = 16000  # Speech Commands convention; adjust if the printed cfg above says otherwise
device = next(model.parameters()).device
print("Model device:", device)

dummy_audio = torch.randn(1, sample_rate, device=device)  # 1 second
dummy_len = torch.tensor([sample_rate], device=device)

with torch.no_grad():
    logits = model(input_signal=dummy_audio, input_signal_length=dummy_len)

print("Forward pass OK. Output shape:", logits.shape)
assert logits.shape[-1] == len(model.cfg.labels), "class count mismatch between output and label list"
print("Output class count matches label list length -- OK")

## Export to ONNX and verify the exported graph actually runs

This is the real point of this notebook -- confirming `.export()` isn't just documented but actually produces a usable file, and that the file runs standalone in `onnxruntime` (i.e. without NeMo/PyTorch in the loop), which is closer to how it'll actually run on the Jetson after the ONNX -> TensorRT step.

**Finding from the first run of this notebook:** the exported graph's only input is `audio_signal` with shape `(batch, 64, time)` -- a precomputed log-mel-spectrogram, not raw waveform. NeMo's `.export()` only traces the encoder+decoder; the preprocessor (waveform -> mel features) is excluded from the graph. This mirrors our own custom-CNN pipeline's split (STFT computed in `audio_dsp.py`, separate from the neural net) -- so the eventual Jetson-side deployment will need its own feature-extraction step before calling the ONNX/TensorRT engine, same shape of problem we already solved once for the custom model. Cross-checking below accounts for this by calling `model.preprocessor(...)` explicitly to get the same mel features the full model uses internally, rather than feeding raw waveform to the ONNX session.

In [ ]:
import os

onnx_path = "matchboxnet_3x1x64_v2.onnx"
model.export(onnx_path)

size_kb = os.path.getsize(onnx_path) / 1024
print(f"Exported: {onnx_path} ({size_kb:.1f} KB)")

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX graph structurally valid.")
print("Graph inputs:")
for inp in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {dims}")
print("Graph outputs:")
for out in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {dims}")

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_names = [i.name for i in session.get_inputs()]
print("\nRuntime input names:", input_names)

# The exported graph takes precomputed mel features (batch, 64, time), not
# raw waveform -- reproduce the same preprocessing step the full model's
# forward() uses internally, so the ONNX inputs actually match what the
# graph expects.
with torch.no_grad():
    processed_signal, processed_signal_len = model.preprocessor(
        input_signal=dummy_audio, length=dummy_len
    )
print("Preprocessed feature shape:", processed_signal.shape)

ort_inputs = {
    session.get_inputs()[0].name: processed_signal.detach().cpu().numpy().astype(np.float32),
}

ort_outputs = session.run(None, ort_inputs)
print("onnxruntime inference OK. Output shape:", ort_outputs[0].shape)

## Cross-check: does the ONNX export match the original PyTorch model's output?

Cheap and important -- an export that runs but produces different numbers than the source model would silently corrupt everything downstream (same category of bug as the Conv/BN-ordering mistake caught earlier in this plan for the custom CNN).

In [ ]:
max_abs_diff = np.max(np.abs(logits.detach().cpu().numpy() - ort_outputs[0]))
print(f"Max abs diff, PyTorch vs ONNX Runtime: {max_abs_diff:.6f}")
assert max_abs_diff < 1e-3, "ONNX export diverges from the source PyTorch model -- do not trust this export"
print("ONNX export matches the source model. Export path verified end-to-end.")

## Summary / decision point

If everything above passed: the deployment path (NeMo checkpoint -> ONNX -> [TensorRT on actual Jetson hardware, not verified here]) is real and functional, so Phase 1 (fine-tuning on our own dataset) is worth investing in.

Still NOT verified by this notebook, and needed before any deployment decision:
- Actual TensorRT engine build + latency/power measurement on real Jetson Orin Nano hardware (or an accurate JetPack-level simulation) -- ONNX validity doesn't guarantee acceptable on-device latency
- Whether the `EncDecClassificationModel`'s expected input format (whole-clip classification) can be adapted cleanly into the existing rolling-buffer streaming receiver loop, or needs a wrapper
- Fine-tuned accuracy/live-stream performance on our actual 12-class dataset -- this checkpoint is pretrained on a different 35-class vocabulary, fine-tuning is still the unproven step